In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

from src.preprocessing import build_vocab
from src.dataset import DisasterTweetsDataset, collate_batch, collate_test_batch
from src.evaluate import evaluate
from src.utils import seed_everything
from src.models import SimpleRNNClassifier, GRUClassifier

In [5]:
df = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [6]:
df["text"].astype(str).str.len().max()

np.int64(157)

In [3]:
print(torch.__version__)

2.12.1+cpu


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

seed_everything(42)
device

device(type='cpu')

In [7]:
# TRAIN TEST SPLIT 
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

len(train_df), len(val_df)

(6090, 1523)

LOAD BERTWEET TOKENIZER

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

CREATE PYTORCH DATASET

In [ ]:
import torch
from torch.utils.data import Dataset


class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

        return item

CREATE DATALOADERS

In [ ]:
from torch.utils.data import DataLoader

MAX_LENGTH = 128
BATCH_SIZE = 16

train_dataset = TweetDataset(
    texts=train_df["input_text"],
    labels=train_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = TweetDataset(
    texts=val_df["input_text"],
    labels=val_df["target"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = TweetDataset(
    texts=test["input_text"],
    labels=None,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

LOAD BERTWEET FOR CLASIFICATION

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

MODEL

In [13]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for x, lengths, y in loader:
        x = x.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

TRAIN

In [14]:
num_epochs = 10

best_f1 = 0
best_state = None

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_probs, val_labels = evaluate(
        model,
        val_loader,
        criterion,
        threshold=0.5
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict()

Epoch 01 | train_loss=0.6404 | val_loss=0.5853 | val_f1=0.5367
Epoch 02 | train_loss=0.5173 | val_loss=0.5400 | val_f1=0.6615
Epoch 03 | train_loss=0.4439 | val_loss=0.5630 | val_f1=0.6809
Epoch 04 | train_loss=0.3654 | val_loss=0.5503 | val_f1=0.7042
Epoch 05 | train_loss=0.2801 | val_loss=0.6273 | val_f1=0.6865
Epoch 06 | train_loss=0.2164 | val_loss=0.6997 | val_f1=0.6932
Epoch 07 | train_loss=0.1646 | val_loss=0.7990 | val_f1=0.7023
Epoch 08 | train_loss=0.1197 | val_loss=0.8861 | val_f1=0.7104
Epoch 09 | train_loss=0.0917 | val_loss=0.9288 | val_f1=0.6969
Epoch 10 | train_loss=0.0811 | val_loss=0.9888 | val_f1=0.7085


In [15]:
model.load_state_dict(best_state)

<All keys matched successfully>